# Implementación de Transformers para Chatbot usando DailyDialog

### Objetivo
En esta evaluación, implementaremos un modelo basado en arquitectura Transformer para una tarea de **diálogo generativo (chatbot)**, utilizando el dataset **DailyDialog**. El modelo será capaz de recibir un mensaje de texto y generar una respuesta coherente.

Usaremos TensorFlow/Keras para construir un modelo Transformer con:
- **Encoder-Decoder**: para procesar la entrada y generar salida secuencial.
- **Atención Multi-cabezal**: para capturar dependencias a largo plazo en el diálogo.

Al final, evaluaremos el modelo usando métricas BLEU y ROUGE, y probaremos el chatbot con mensajes propios.

## 1. Instalación de dependencias y carga del dataset

In [ ]:
# Instalar dependencias necesarias
!pip install datasets nltk rouge-score --quiet

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Embedding, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)
from datasets import load_dataset
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import re
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Librerías cargadas correctamente")

In [ ]:
# Cargar DailyDialog desde Hugging Face
dataset = load_dataset("OpenRL/daily_dialog")

print("--- TAMAÑO DEL DATASET ---")
print(f"Entrenamiento: {dataset['train'].num_rows} diálogos")
print(f"Validación: {dataset['validation'].num_rows} diálogos")
print(f"Prueba: {dataset['test'].num_rows} diálogos")

# Mostrar un ejemplo
ejemplo = dataset['train'][0]
print("\n--- EJEMPLO DE DIÁLOGO ---")
for i, turno in enumerate(ejemplo['dialog']):
    print(f"  Turno {i+1}: {turno}")

### 1.1 Análisis exploratorio del dataset

In [ ]:
# Convertir a pandas para análisis
df_train = dataset['train'].to_pandas()

df_train['num_turns'] = df_train['dialog'].apply(len)
print("--- ESTADÍSTICAS: TURNOS POR DIÁLOGO ---")
print(df_train['num_turns'].describe())

# Distribución de emociones
df_frases = df_train.explode(['dialog', 'emotion', 'act'])
emotion_map = {0: 'Neutral', 1: 'Anger', 2: 'Disgust', 3: 'Fear',
               4: 'Happiness', 5: 'Sadness', 6: 'Surprise'}
act_map = {1: 'Inform', 2: 'Question', 3: 'Directive', 4: 'Commissive'}

df_frases['emotion_name'] = df_frases['emotion'].map(emotion_map)
df_frases['act_name'] = df_frases['act'].map(act_map)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.countplot(data=df_frases, x='emotion_name',
              order=df_frases['emotion_name'].value_counts().index,
              ax=axes[0], palette='viridis')
axes[0].set_title('Distribución de Emociones')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(data=df_frases, x='act_name',
              order=df_frases['act_name'].value_counts().index,
              ax=axes[1], palette='magma')
axes[1].set_title('Distribución de Actos de Diálogo')

plt.tight_layout()
plt.show()
print(f"\nTotal de frases (pares potenciales): {len(df_frases)}")

## 2. Preprocesamiento de datos para entrenamiento

Convertiremos cada diálogo en pares (entrada -> respuesta). Por ejemplo, si un diálogo tiene N turnos, creamos N-1 pares donde el contexto es el turno i y la respuesta es el turno i+1.

In [ ]:
# Crear pares de entrenamiento (input, target)
def create_pairs(dialog):
    pairs = []
    for i in range(len(dialog) - 1):
        inp = dialog[i].strip()
        tgt = dialog[i + 1].strip()
        if inp and tgt:
            pairs.append((inp, tgt))
    return pairs

train_pairs = []
for dialog in dataset['train']['dialog']:
    train_pairs.extend(create_pairs(dialog))

val_pairs = []
for dialog in dataset['validation']['dialog']:
    val_pairs.extend(create_pairs(dialog))

test_pairs = []
for dialog in dataset['test']['dialog']:
    test_pairs.extend(create_pairs(dialog))

print(f"Pares de entrenamiento: {len(train_pairs)}")
print(f"Pares de validación: {len(val_pairs)}")
print(f"Pares de prueba: {len(test_pairs)}")

print("\n--- EJEMPLOS DE PARES (input -> target) ---")
for i in range(3):
    print(f"  [{i}] INPUT: {train_pairs[i][0]}")
    print(f"      TARGET: {train_pairs[i][1]}")
    print()

In [ ]:
# Limpieza bÃ¡sica de texto
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9?.!,Â¿Â¡'\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_pairs = [(clean_text(inp), clean_text(tgt)) for inp, tgt in train_pairs]
val_pairs = [(clean_text(inp), clean_text(tgt)) for inp, tgt in val_pairs]
test_pairs = [(clean_text(inp), clean_text(tgt)) for inp, tgt in test_pairs]

# Verificar longitud de frases
input_lens = [len(inp.split()) for inp, _ in train_pairs]
target_lens = [len(tgt.split()) for _, tgt in train_pairs]

print(f"Longitud input - media: {np.mean(input_lens):.1f}, max: {max(input_lens)}, p95: {np.percentile(input_lens, 95):.0f}")
print(f"Longitud target - media: {np.mean(target_lens):.1f}, max: {max(target_lens)}, p95: {np.percentile(target_lens, 95):.0f}")

MAX_LEN = 30

### 2.1 TokenizaciÃ³n y creaciÃ³n de vocabulario

In [ ]:
# Construir vocabulario usando tf_text
all_sentences = [inp for inp, _ in train_pairs] + [tgt for _, tgt in train_pairs]

# Tokenizer con keras
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=15000,
    output_sequence_length=MAX_LEN,
    standardize='lower_and_strip_punctuation',
    split='whitespace'
)

vectorizer.adapt(all_sentences)

VOCAB_SIZE = len(vectorizer.get_vocabulary())
print(f"TamaÃ±o del vocabulario: {VOCAB_SIZE}")
print(f"Tokens especiales: {vectorizer.get_vocabulary()[:10]}")

In [ ]:
# Crear dataset de TensorFlow
def encode_pair(inp, tgt):
    inp_ids = vectorizer(inp)
    tgt_ids = vectorizer(tgt)
    return inp_ids, tgt_ids

def make_dataset(pairs, batch_size=64, shuffle=True):
    inputs = [p[0] for p in pairs]
    targets = [p[1] for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((inputs, targets))
    ds = ds.map(lambda inp, tgt: encode_pair(inp, tgt),
                num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(10000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

BATCH_SIZE = 64
train_ds = make_dataset(train_pairs, BATCH_SIZE)
val_ds = make_dataset(val_pairs, BATCH_SIZE, shuffle=False)
test_ds = make_dataset(test_pairs, BATCH_SIZE, shuffle=False)

# Verificar forma de los datos
for inp_batch, tgt_batch in train_ds.take(1):
    print(f"Batch input shape: {inp_batch.shape}")
    print(f"Batch target shape: {tgt_batch.shape}")
    break

## 3. ImplementaciÃ³n del Modelo Transformer

Construimos un modelo Transformer completo con Encoder-Decoder y atenciÃ³n multi-cabezal.

In [ ]:
# Capa de positional encoding
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

    def call(self, x):
        seq_len = tf.shape(x)[1]
        position = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)
        div_term = tf.exp(tf.cast(tf.range(0, self.d_model, 2), tf.float32) *
                          -(tf.math.log(10000.0) / self.d_model))
        pe_sin = tf.sin(position * div_term)
        pe_cos = tf.cos(position * div_term)
        pe = tf.concat([pe_sin, pe_cos], axis=-1)
        pe = pe[tf.newaxis, :, :]
        return x + pe


# Capa del Encoder
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(d_model)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, x, training, mask=None):
        attn_output = self.attn(x, x, attention_mask=mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


# Capa del Decoder
class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.attn1 = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.attn2 = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(d_model)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.layernorm3 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)
        self.dropout3 = Dropout(rate)

    def call(self, x, enc_output, training,
             look_ahead_mask=None, padding_mask=None):
        attn1 = self.attn1(x, x, attention_mask=look_ahead_mask)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layernorm1(x + attn1)
        attn2 = self.attn2(out1, enc_output, attention_mask=padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layernorm2(out1 + attn2)
        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        return self.layernorm3(out2 + ffn_output)

In [ ]:
# Modelo Transformer completo
class Transformer(tf.keras.Model):
    def __init__(self, vocab_size, d_model, num_heads, ff_dim,
                 num_encoder_layers, num_decoder_layers, max_len, rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = Embedding(vocab_size, d_model, mask_zero=True)
        self.pos_encoding = PositionalEncoding(d_model)

        self.encoder_layers = [
            TransformerEncoder(d_model, num_heads, ff_dim, rate)
            for _ in range(num_encoder_layers)
        ]
        self.decoder_layers = [
            TransformerDecoder(d_model, num_heads, ff_dim, rate)
            for _ in range(num_decoder_layers)
        ]

        self.final_layer = Dense(vocab_size)
        self.max_len = max_len

    def call(self, inputs, training=False):
        inp, tar = inputs

        enc_padding_mask = self.create_padding_mask(inp)
        dec_padding_mask = self.create_padding_mask(tar)
        look_ahead_mask = self.create_look_ahead_mask(tf.shape(tar)[1])
        combined_mask = tf.maximum(dec_padding_mask, look_ahead_mask)

        enc_emb = self.embedding(inp) * tf.math.sqrt(
            tf.cast(self.d_model, tf.float32))
        enc_emb = self.pos_encoding(enc_emb)

        dec_emb = self.embedding(tar) * tf.math.sqrt(
            tf.cast(self.d_model, tf.float32))
        dec_emb = self.pos_encoding(dec_emb)

        enc_output = enc_emb
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, training, enc_padding_mask)

        dec_output = dec_emb
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, training,
                                  combined_mask, dec_padding_mask)

        output = self.final_layer(dec_output)
        return output

    def create_padding_mask(self, seq):
        mask = tf.cast(tf.equal(seq, 0), tf.float32)
        return mask[:, tf.newaxis, tf.newaxis, :]

    def create_look_ahead_mask(self, size):
        mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
        return mask[tf.newaxis, tf.newaxis, :, :]


# HiperparÃ¡metros
D_MODEL = 256
NUM_HEADS = 4
FF_DIM = 512
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
DROPOUT_RATE = 0.1

transformer = Transformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    max_len=MAX_LEN,
    rate=DROPOUT_RATE
)

transformer.summary()

### 3.1 FunciÃ³n de pÃ©rdida y optimizador

In [ ]:
# FunciÃ³n de pÃ©rdida con masking de padding
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.cast(tf.not_equal(real, 0), tf.float32)
    loss_ = loss_object(real, pred)
    loss_ *= mask
    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)


# MÃ©tricas
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.Mean(name='train_accuracy')


# Optimizador con learning rate schedule (como en el paper original)
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.minimum(arg1, arg2)


learning_rate = CustomSchedule(D_MODEL)
optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98,
                                     epsilon=1e-9)

## 4. Entrenamiento del Modelo

In [ ]:
# Paso de entrenamiento
@tf.function
def train_step(inp, tar):
    tar_inp = tar[:, :-1]
    tar_real = tar[:, 1:]

    with tf.GradientTape() as tape:
        predictions = transformer((inp, tar_inp), training=True)
        loss = loss_function(tar_real, predictions)

    gradients = tape.gradient(loss, transformer.trainable_variables)
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))

    train_loss(loss)
    train_accuracy(accuracy_function(tar_real, predictions))


def accuracy_function(real, pred):
    accuracies = tf.equal(real, tf.cast(tf.argmax(pred, axis=-1), tf.int32))
    mask = tf.cast(tf.not_equal(real, 0), tf.float32)
    accuracies = tf.cast(accuracies, tf.float32) * mask
    return tf.reduce_sum(accuracies) / tf.reduce_sum(mask)

In [ ]:
# Bucle de entrenamiento
EPOCHS = 10
history = {'loss': [], 'val_loss': [], 'accuracy': []}

for epoch in range(EPOCHS):
    train_loss.reset_state()
    train_accuracy.reset_state()

    for batch, (inp, tar) in enumerate(train_ds):
        train_step(inp, tar)
        if batch % 100 == 0:
            print(f'Epoch {epoch + 1} Batch {batch} '
                  f'Loss {train_loss.result():.4f} '
                  f'Accuracy {train_accuracy.result():.4f}')

    # ValidaciÃ³n
    val_loss = 0
    val_steps = 0
    for inp, tar in val_ds:
        tar_inp = tar[:, :-1]
        tar_real = tar[:, 1:]
        predictions = transformer((inp, tar_inp), training=False)
        val_loss += loss_function(tar_real, predictions)
        val_steps += 1

    val_loss_avg = val_loss / val_steps
    train_loss_val = train_loss.result()
    train_acc_val = train_accuracy.result()

    history['loss'].append(float(train_loss_val))
    history['val_loss'].append(float(val_loss_avg))
    history['accuracy'].append(float(train_acc_val))

    print(f'\nEpoch {epoch + 1} - '
          f'Train Loss: {train_loss_val:.4f}, '
          f'Val Loss: {val_loss_avg:.4f}, '
          f'Train Acc: {train_acc_val:.4f}\n')

In [ ]:
# Graficar evoluciÃ³n del entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Curva de PÃ©rdida')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['accuracy'], label='Train Accuracy', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('PrecisiÃ³n en Entrenamiento')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 5. EvaluaciÃ³n del Modelo

Evaluamos usando las mÃ©tricas **BLEU** y **ROUGE** sobre el conjunto de prueba.

In [ ]:
# FunciÃ³n para generar respuesta (inferencia)
def generate_response(model, input_text, max_len=MAX_LEN):
    input_text = clean_text(input_text)
    input_ids = vectorizer([input_text])

    start_token = tf.constant([[2]])
    output = tf.cast(start_token, tf.int64)

    for _ in range(max_len):
        predictions = model((input_ids, output), training=False)
        predictions = predictions[:, -1:, :]
        predicted_id = tf.cast(tf.argmax(predictions, axis=-1), tf.int64)
        if tf.equal(predicted_id[0][0], 0):
            break
        output = tf.concat([output, predicted_id], axis=-1)

    output = tf.squeeze(output, axis=0)
    words = []
    vocab = vectorizer.get_vocabulary()
    for token in output.numpy():
        if token == 0:
            break
        words.append(vocab[token])
    return ' '.join(words[1:])


# Mostrar ejemplos de generaciÃ³n
print("--- EJEMPLOS DE RESPUESTAS GENERADAS ---\n")
for i, (inp, real_tgt) in enumerate(test_pairs[:5]):
    pred = generate_response(transformer, inp)
    print(f"Input: {inp}")
    print(f"Real:  {real_tgt}")
    print(f"Pred:  {pred}")
    print()

In [ ]:
# Calcular BLEU en un subconjunto de test
smoothie = SmoothingFunction().method4

bleu_scores = []
num_eval = min(500, len(test_pairs))

for i in range(num_eval):
    inp, real_tgt = test_pairs[i]
    pred = generate_response(transformer, inp)
    reference = [real_tgt.split()]
    candidate = pred.split()
    if len(candidate) == 0:
        continue
    score = sentence_bleu(reference, candidate,
                         smoothing_function=smoothie)
    bleu_scores.append(score)

print(f"--- MÃ‰TRICA BLEU ---")
print(f"BLEU promedio sobre {len(bleu_scores)} muestras: {np.mean(bleu_scores):.4f}")
print(f"BLEU mÃ¡ximo: {np.max(bleu_scores):.4f}")
print(f"BLEU mÃ­nimo: {np.min(bleu_scores):.4f}")

In [ ]:
# Calcular ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'],
                                  use_stemmer=True)

rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}

for i in range(num_eval):
    inp, real_tgt = test_pairs[i]
    pred = generate_response(transformer, inp)
    if len(pred.split()) == 0:
        continue
    scores = scorer.score(real_tgt, pred)
    for metric in ['rouge1', 'rouge2', 'rougeL']:
        rouge_scores[metric].append(scores[metric].fmeasure)

print("--- MÃ‰TRICAS ROUGE ---")
for metric in ['rouge1', 'rouge2', 'rougeL']:
    print(f"{metric}: {np.mean(rouge_scores[metric]):.4f} (promedio F1)")

## 6. Chatbot Interactivo

Â¡Probemos el chatbot con mensajes propios!

In [ ]:
def chatbot(message):
    response = generate_response(transformer, message)
    if not response:
        response = "(no pude generar una respuesta)"
    return response


# Pruebas con mensajes personalizados
test_messages = [
    "Hello, how are you?",
    "What do you want to do today?",
    "I like playing football",
    "Can you help me with my homework?",
    "Good morning!"
]

print("=== CHATBOT RESPONDE ===\n")
for msg in test_messages:
    resp = chatbot(msg)
    print(f"TÃº: {msg}")
    print(f"Bot: {resp}")
    print()

In [ ]:
# Chat interactivo (descomenta para usar)
# print("=== CHAT INTERACTIVO ===")
# print("Escribe 'quit' para salir.\n")
# while True:
#     msg = input("TÃº: ")
#     if msg.lower() == 'quit':
#         break
#     resp = chatbot(msg)
#     print(f"Bot: {resp}\n")

## 7. Ajuste de HiperparÃ¡metros

Probamos diferentes configuraciones para ver su impacto en rendimiento.

In [ ]:
# FunciÃ³n para crear modelo con hiperparÃ¡metros personalizados
def create_transformer(d_model, num_heads, ff_dim,
                       num_layers, vocab_size, max_len):
    return Transformer(
        vocab_size=vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_encoder_layers=num_layers,
        num_decoder_layers=num_layers,
        max_len=max_len,
        rate=0.1
    )


# Configuraciones a probar
configs = [
    {'name': 'Base', 'd_model': 128, 'num_heads': 2, 'ff_dim': 256, 'layers': 2},
    {'name': 'Medio', 'd_model': 256, 'num_heads': 4, 'ff_dim': 512, 'layers': 2},
    {'name': 'Grande', 'd_model': 256, 'num_heads': 4, 'ff_dim': 512, 'layers': 4},
    {'name': 'Cabezas+', 'd_model': 256, 'num_heads': 8, 'ff_dim': 512, 'layers': 2},
]

print("Configuraciones a probar:")
for cfg in configs:
    print(f"  {cfg['name']}: d_model={cfg['d_model']}, heads={cfg['num_heads']}, "
          f"ff_dim={cfg['ff_dim']}, layers={cfg['layers']}")

In [ ]:
# NOTA: Esta celda es referencial - el entrenamiento completo de cada
# configuraciÃ³n tomarÃ­a mucho tiempo.

print("Para hacer tuning completo, iterar sobre configuraciones:")
print("""
for cfg in configs:
    model = create_transformer(
        cfg['d_model'], cfg['num_heads'], cfg['ff_dim'],
        cfg['layers'], VOCAB_SIZE, MAX_LEN
    )
    # Entrenar 1-2 epochs y evaluar BLEU
""")

print("\n--- RESULTADOS ESPERADOS (hipotÃ©ticos) ---")
print("| Config | Params | BLEU | Tiempo/epoch |")
print("|--------|--------|------|-------------|")
print("|  Base  |   ~1M  | 0.02 |   ~3 min     |")
print("|  Medio |   ~4M  | 0.03 |   ~5 min     |")
print("| Grande |   ~8M  | 0.03 |   ~10 min    |")
print("|Cabezas+|   ~4M  | 0.03 |   ~6 min     |")

## 8. PresentaciÃ³n de Resultados y Conclusiones

### Resultados Finales

| Modelo | HiperparÃ¡metros | BLEU | ROUGE-1 | ROUGE-L |
|--------|----------------|------|---------|---------|
| Transformer | d_model=256, heads=4, ff_dim=512, layers=2 | X.XXXX | X.XXXX | X.XXXX |

### InterpretaciÃ³n

- **BLEU**: Mide la similitud exacta de n-gramas entre la respuesta generada y la referencia.
- **ROUGE**: EvalÃºa la cobertura de contenido entre la respuesta generada y la referencia.
- Un score bajo no significa que el modelo funcione mal, sino que la mÃ©trica de n-gramas exactos es muy restrictiva para diÃ¡logo abierto.

### AnÃ¡lisis de HiperparÃ¡metros

- **d_model**: A mayor dimensiÃ³n, mayor capacidad pero mÃ¡s parÃ¡metros.
- **num_heads**: MÃ¡s cabezas permiten atender a diferentes relaciones contextuales.
- **ff_dim**: Controla la capacidad de transformaciÃ³n no lineal.
- **num_layers**: Modelos mÃ¡s profundos capturan jerarquÃ­as mÃ¡s complejas.

### Dificultades encontradas

1. **Preprocesamiento**: Los diÃ¡logos tienen longitudes variables.
2. **Recursos computacionales**: El entrenamiento de transformers es intensivo.
3. **EvaluaciÃ³n de diÃ¡logo**: Las mÃ©tricas BLEU/ROUGE no capturan bien la calidad de un chatbot.

### Aprendizajes

- La arquitectura Transformer permite generar texto coherente y contextualizado.
- La atenciÃ³n multi-cabezal es clave para capturar relaciones en el diÃ¡logo.
- DailyDialog es un excelente dataset para tareas de diÃ¡logo por su variedad de temas y emociones.